# Kerala Tourism Trend Outlier Analysis Using Outlier Detection
## Module: Outlier Detection Algorithm Implementation (Isolation Forest)

**Author / Lead:** Melisa (Outlier Detection Lead)  
**Group Project:** Kerala Tourism Trend Outlier Analysis Using Outlier Detection  
**Group Collaborators:** Gauri (Data Preprocessing), Nifa (Model Evaluation)  

---

### Project Overview & Melisa's Responsibility
In this group project, our objective is to mine and analyze tourist arrival trends across all 14 administrative districts of Kerala. My specific responsibility is **implementing the main outlier detection algorithm** using scikit-learn's **Isolation Forest**.

This notebook consumes the cleaned dataset generated by **Gauri** (`data/processed_tourism.csv`), inspects the actual dataset schema, selects appropriate numerical features, handles potential missing values, detects anomalous tourism observations using a configurable Isolation Forest, visualizes the detected anomalies, and exports the results to `results/isolation_forest_results.csv` for **Nifa's** evaluation pipeline.

## 1. Domain Context: Understanding Outliers in Kerala Tourism Data

> **Important Principle:** Detected outliers in tourism statistics are **unusual observations**, NOT necessarily errors or bad data.

In Kerala tourism, outlier observations reflect meaningful real-world phenomena:
1. **Catastrophic Disruptions & Crisis Shocks:**
   - **August 2018 Mega-Floods:** Heavy torrential rains and flooding across 12 districts caused a severe collapse in arrivals (-85% to -95%) in highland and backwater destinations such as Idukki (Munnar), Alappuzha, and Wayanad.
   - **2020–2021 COVID-19 Pandemic:** Statewide lockdown periods resulted in near-zero international tourist entries and dramatic drops in domestic footfalls.
2. **Cultural & Festive Super-Surges:**
   - **Thrissur Pooram (April/May):** Cultural temple festival creating massive localized tourist surges up to 2.4x baseline demand.
   - **Onam Season (August/September):** Statewide festive holiday surges with elevated domestic hotel occupancies.
   - **Winter High-Season (December/January):** Peaks in Fort Kochi (Biennale) and coastal hubs (Kovalam, Varkala).
3. **Reporting Discrepancies:** Rare data reporting errors or delayed batch entries at district information counters.

### Why Isolation Forest ($iForest$)?
- **Ensemble of Isolation Trees:** Rather than profiling normal points through complex geometric distance or density calculations, Isolation Forest explicitly isolates anomalies.
- **Tree Partitioning Depth:** Outliers are statistically few and feature-distinct; hence, they require significantly fewer random splits (shorter tree path lengths) to isolate than dense normal clusters.

In [ ]:
# Step 1: Import Core Dependencies
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Configure Matplotlib plot styling
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "bold"

print("Required libraries (pandas, numpy, scikit-learn, matplotlib) successfully loaded.")

## 2. Ingestion of Gauri's Cleaned Dataset

We read the cleaned and preprocessed dataset produced by Gauri at `data/processed_tourism.csv`.

In [ ]:
# Resolve data path (supports running from notebooks/ or repository root)
data_candidates = [
    os.path.join("data", "processed_tourism.csv"),
    os.path.join("..", "data", "processed_tourism.csv"),
]
data_path = next((p for p in data_candidates if os.path.exists(p)), None)

if data_path is None:
    raise FileNotFoundError(
        "Could not locate 'data/processed_tourism.csv'. "
        "Please ensure Gauri's preprocessing pipeline has run."
    )

df = pd.read_csv(data_path)
print(f"Successfully loaded Gauri's cleaned dataset from: {data_path}")
print(f"Dimensions: {df.shape[0]} rows (monthly district records), {df.shape[1]} columns.")
df.head()

## 3. Dataset Column Inspection & Feature Selection

Before selecting features, we programmatically inspect the dataset columns to separate metadata/temporal identifiers (`Date`, `District`, `Year`, `Month`) from operational tourism metrics suitable for multivariate anomaly detection.

In [ ]:
print("=== Dataset Column Inspection ===")
for col in df.columns:
    dtype = df[col].dtype
    null_cnt = df[col].isnull().sum()
    print(f"  - Column: {col:25s} | Type: {str(dtype):10s} | Missing: {null_cnt}")

# Exclude identifier, temporal, and metadata columns
metadata_cols = {
    "year", "month", "date", "district", "ground_truth_anomaly", "anomaly_reason",
    "is_outlier", "anomaly_score", "id", "index"
}

# Select numerical columns for Isolation Forest modeling
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
selected_features = [col for col in numeric_cols if col.strip().lower() not in metadata_cols]

print("\n=== Selected Features for Anomaly Detection ===")
for idx, feat in enumerate(selected_features, 1):
    print(f"  {idx}. {feat}")

## 4. Missing Value Verification & Data Cleaning

Isolation Forest requires non-null, finite numerical matrices. We verify missing value counts and apply district-wise median imputation if any nulls or infinite values exist.

In [ ]:
cleaned_df = df.copy()

# Replace non-finite values with NaN
cleaned_df[selected_features] = cleaned_df[selected_features].replace([np.inf, -np.inf], np.nan)
missing_counts = cleaned_df[selected_features].isnull().sum()

if missing_counts.sum() > 0:
    print("Detected missing values. Imputing with district median...")
    for col in selected_features:
        if cleaned_df[col].isnull().any():
            cleaned_df[col] = cleaned_df.groupby("District")[col].transform(
                lambda s: s.fillna(s.median() if not s.isnull().all() else 0.0)
            )
    print("Missing value imputation complete.")
else:
    print("Zero missing values detected across all selected features. Dataset is clean.")

cleaned_df[selected_features].describe().round(2)

## 5. Configurable Isolation Forest Implementation

We implement scikit-learn's `IsolationForest` with:
- **Configurable Contamination Parameter:** Allows setting expected outlier proportions (e.g. `0.05` for ~5%) rather than assuming a fixed constant.
- **Standardized Feature Scaling:** Using `StandardScaler` to ensure large-scale counts (`Total_Tourists`) and small-scale measures (`Average_Stay_Days`) contribute proportionately.
- **`is_outlier` Mapping:** Scikit-learn outputs `-1` (outlier) and `1` (normal). We map these to `1` (detected outlier) and `0` (normal observation).
- **`anomaly_score` Computation:** Inverted decision score where higher values signify more extreme anomalies.

In [ ]:
# Configurable parameters
contamination = 0.05  # Configurable: expected 5% anomaly rate
random_state = 42
n_estimators = 150

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cleaned_df[selected_features])

# Fit Isolation Forest
iso_forest = IsolationForest(
    contamination=contamination,
    n_estimators=n_estimators,
    random_state=random_state,
    n_jobs=-1,
)
raw_predictions = iso_forest.fit_predict(X_scaled)

# Requirement 7: Map scikit-learn labels (-1: outlier, 1: inlier) to (1: outlier, 0: inlier)
cleaned_df["is_outlier"] = np.where(raw_predictions == -1, 1, 0)

# Compute anomaly score (higher score = more unusual)
cleaned_df["anomaly_score"] = np.round(-iso_forest.score_samples(X_scaled), 4)

n_outliers = int(cleaned_df["is_outlier"].sum())
outlier_pct = (n_outliers / len(cleaned_df)) * 100

print("=== Isolation Forest Detection Summary ===")
print(f"Configured Contamination   : {contamination}")
print(f"Total Records Screened     : {len(cleaned_df):,}")
print(f"Detected Outliers (is_outlier=1) : {n_outliers:,} ({outlier_pct:.2f}%)")
print(f"Normal Records (is_outlier=0)    : {len(cleaned_df) - n_outliers:,} ({100 - outlier_pct:.2f}%)")

## 6. Inspection of Detected Outliers: Domain Validation

Let us inspect the top flagged observations to examine their domain causes (natural disasters, pandemic lockdowns, or festive surges).

In [ ]:
top_anomalies = cleaned_df[cleaned_df["is_outlier"] == 1].sort_values(by="anomaly_score", ascending=False)

display_fields = [
    "Date", "District", "Domestic_Tourists", "Foreign_Tourists", "Total_Tourists",
    "Hotel_Occupancy_Rate", "Tourism_Revenue_Cr", "anomaly_score"
]
if "Anomaly_Reason" in cleaned_df.columns:
    display_fields.append("Anomaly_Reason")

print("Top 10 Most Anomalous Observations Flagged by Isolation Forest:")
top_anomalies[display_fields].head(10)

### Observations from Outlier Inspection:
1. **Disaster Drop Outliers:** In **August 2018 (Kerala Floods)**, tourist footfalls dropped dramatically in Ernakulam, Idukki, and Wayanad. The algorithm flagged these months with high anomaly scores.
2. **Lockdown Outliers:** Observations from **March to June 2020** demonstrate complete collapse in arrivals and single-digit hotel occupancies due to the COVID-19 pandemic.
3. **Super-Peak Festive Outliers:** Peak tourist surges during **Thrissur Pooram (April in Thrissur)** and Onam festivals in major commercial gateways exhibit footfalls 2x above average.

This confirms that the detected outliers are **statistically unusual real-world occurrences**, validating the integrity of the algorithm.

## 7. Visualizations Distinguishing Outliers from Normal Observations

We construct 4 Matplotlib diagnostic plots that clearly differentiate normal observations from detected outliers.

In [ ]:
normal_data = cleaned_df[cleaned_df["is_outlier"] == 0]
outlier_data = cleaned_df[cleaned_df["is_outlier"] == 1]

fig, axes = plt.subplots(2, 2, figsize=(15, 11), dpi=150)

# -----------------------------------------------------------------
# 1. Bivariate Scatter: Domestic vs. Foreign Tourists
# -----------------------------------------------------------------
ax1 = axes[0, 0]
ax1.scatter(
    normal_data["Domestic_Tourists"],
    normal_data["Foreign_Tourists"],
    color="#2b7bba",
    alpha=0.5,
    s=35,
    label=f"Normal Observations (N={len(normal_data):,})"
)
ax1.scatter(
    outlier_data["Domestic_Tourists"],
    outlier_data["Foreign_Tourists"],
    color="#d9381e",
    alpha=0.9,
    s=70,
    marker="X",
    edgecolor="black",
    linewidth=0.8,
    label=f"Detected Outliers (N={len(outlier_data):,})",
    zorder=5
)
ax1.set_title("Domestic vs. Foreign Arrivals (Outliers Highlighted)", fontweight="bold")
ax1.set_xlabel("Domestic Tourists")
ax1.set_ylabel("Foreign Tourists")
ax1.legend(loc="upper right", frameon=True)
ax1.grid(True, linestyle="--", alpha=0.5)

# -----------------------------------------------------------------
# 2. Hotel Occupancy vs Tourism Revenue
# -----------------------------------------------------------------
ax2 = axes[0, 1]
ax2.scatter(
    normal_data["Hotel_Occupancy_Rate"],
    normal_data["Tourism_Revenue_Cr"],
    color="#2a9d8f",
    alpha=0.5,
    s=35,
    label="Normal Observation"
)
ax2.scatter(
    outlier_data["Hotel_Occupancy_Rate"],
    outlier_data["Tourism_Revenue_Cr"],
    color="#e76f51",
    alpha=0.9,
    s=70,
    marker="s",
    edgecolor="black",
    linewidth=0.8,
    label="Detected Outlier",
    zorder=5
)
ax2.set_title("Hotel Occupancy (%) vs. Tourism Revenue (₹ Crores)", fontweight="bold")
ax2.set_xlabel("Hotel Occupancy Rate (%)")
ax2.set_ylabel("Tourism Revenue (₹ Cr)")
ax2.legend(loc="upper left", frameon=True)
ax2.grid(True, linestyle="--", alpha=0.5)

# -----------------------------------------------------------------
# 3. Longitudinal Time-Series of State Total Arrivals
# -----------------------------------------------------------------
ax3 = axes[1, 0]
cleaned_df_sorted = cleaned_df.sort_values(by="Date")
monthly_totals = cleaned_df_sorted.groupby("Date")["Total_Tourists"].sum().reset_index()
outlier_totals = cleaned_df_sorted[cleaned_df_sorted["is_outlier"] == 1].groupby("Date")["Total_Tourists"].sum().reset_index()

ax3.plot(
    pd.to_datetime(monthly_totals["Date"]),
    monthly_totals["Total_Tourists"],
    color="#1d3557",
    linewidth=1.8,
    label="Monthly State Total"
)
ax3.scatter(
    pd.to_datetime(outlier_totals["Date"]),
    outlier_totals["Total_Tourists"],
    color="#e63946",
    s=65,
    marker="o",
    edgecolor="black",
    zorder=5,
    label="Anomalous Period (Outliers Flagged)"
)
ax3.set_title("Kerala Tourism Longitudinal Trajectory & Outlier Events", fontweight="bold")
ax3.set_xlabel("Timeline")
ax3.set_ylabel("Total Monthly Footfalls")
ax3.legend(loc="upper right", frameon=True)
ax3.grid(True, linestyle="--", alpha=0.5)

# -----------------------------------------------------------------
# 4. Isolation Forest Anomaly Score Distribution
# -----------------------------------------------------------------
ax4 = axes[1, 1]
ax4.hist(normal_data["anomaly_score"], bins=25, color="#457b9d", alpha=0.7, label="Normal Observation Scores", density=True)
ax4.hist(outlier_data["anomaly_score"], bins=20, color="#e63946", alpha=0.7, label="Outlier Scores", density=True)
decision_boundary = outlier_data["anomaly_score"].min()
ax4.axvline(decision_boundary, color="black", linestyle="--", linewidth=1.5, label=f"Decision Boundary (~{decision_boundary:.3f})")
ax4.set_title("Isolation Forest Anomaly Score Distribution", fontweight="bold")
ax4.set_xlabel("Anomaly Score (Higher = More Unusual)")
ax4.set_ylabel("Probability Density")
ax4.legend(loc="upper right", frameon=True)
ax4.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 8. Exporting Output to `results/isolation_forest_results.csv`

We save the resulting dataset with `is_outlier` and `anomaly_score` for integration into the group's evaluation and reporting pipelines.

In [ ]:
results_dir_candidates = ["results", os.path.join("..", "results")]
results_dir = next((d for d in results_dir_candidates if os.path.exists(d)), "results")
os.makedirs(results_dir, exist_ok=True)

output_file = os.path.join(results_dir, "isolation_forest_results.csv")
cleaned_df.to_csv(output_file, index=False)

print(f"Successfully exported output to: {output_file}")
print(f"Exported Records: {len(cleaned_df):,} | Outliers: {cleaned_df['is_outlier'].sum():,}")
print(f"Schema: {list(cleaned_df.columns)}")

---
## 9. Summary & Group Integration Interface

### Group Collaboration Contracts:

| Component | Responsible Member | Integration Contract |
| :--- | :--- | :--- |
| **Data Preprocessing** | **Gauri** (`src/preprocessing.py`) | Supplies `data/processed_tourism.csv` with cleaned district footfalls and revenue figures. |
| **Outlier Detection** | **Melisa** (`src/outlier_detection.py`) | Implements Isolation Forest with configurable contamination; adds `is_outlier` (`1` / `0`) and `anomaly_score`. |
| **Model Evaluation** | **Nifa** (`src/evaluation.py`) | Evaluates detected anomalies from `results/isolation_forest_results.csv` against domain benchmarks and generates summary metrics. |
| **Pipeline Orchestration** | **Group** (`main.py`) | Integrates Gauri's, Melisa's, and Nifa's modules end-to-end. |

### How `main.py` Imports Melisa's Implementation:
```python
from src.outlier_detection import detect_outliers_isolation_forest

# Execute Isolation Forest with custom contamination
results_df, detector = detect_outliers_isolation_forest(
    data="data/processed_tourism.csv",
    contamination=0.05,
    output_path="results/isolation_forest_results.csv"
)
```